In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os

# Loading Raw ACS Data into a DataFrame

In [2]:
df = pd.read_csv("../data/acsd_raw.csv")

# Exploratory Data Analysis

Using a Custom Function from the Utilities Notebook

In [3]:
from utilities import basic_eda

basic_eda(df)

DataFrame Shape: (224220, 13)

 Column Names:
['SERIALNO', 'OCCP', 'AGEP', 'SEX', 'RAC1P', 'HISP', 'SCHL', 'STATE', 'PUMA', 'ESR', 'WAGP', 'COW', 'WKHP']

 Data Types:
SERIALNO     object
OCCP        float64
AGEP          int64
SEX           int64
RAC1P         int64
HISP          int64
SCHL        float64
STATE         int64
PUMA          int64
ESR         float64
WAGP        float64
COW         float64
WKHP        float64
dtype: object

 DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224220 entries, 0 to 224219
Data columns (total 13 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   SERIALNO  224220 non-null  object 
 1   OCCP      126524 non-null  float64
 2   AGEP      224220 non-null  int64  
 3   SEX       224220 non-null  int64  
 4   RAC1P     224220 non-null  int64  
 5   HISP      224220 non-null  int64  
 6   SCHL      217649 non-null  float64
 7   STATE     224220 non-null  int64  
 8   PUMA      224220 non-nul

# Cleaning

## Column Names

Using the data dictionary provided for this dataset, I will start by changing column names to descriptive ones.

In [4]:
column_mapping = {
    'SERIALNO': 'serial_number',
    'OCCP': 'occupation',
    'AGEP': 'age',
    'SEX': 'sex',
    'RAC1P': 'race',
    'HISP': 'hispanic_origin',
    'SCHL': 'education_level',
    'STATE': 'state_fips',
    'PUMA': 'puma_area',
    'ESR': 'employment_status',
    'WAGP': 'wage_income',
    'COW': 'class_of_worker',
    'WKHP': 'hours_per_week'
}

df.rename(columns=column_mapping, inplace=True)

df.head()

,serial_number,occupation,age,sex,race,hispanic_origin,education_level,state_fips,puma_area,employment_status,wage_income,class_of_worker,hours_per_week
0,2019GQ0000143,8740.0,22,1,2,1,17.0,21,1903,6.0,1500.0,1.0,40.0
1,2019GQ0000172,NaN,19,1,1,1,16.0,21,1703,6.0,0.0,NaN,NaN
2,2019GQ0000249,NaN,75,1,1,1,16.0,21,100,6.0,0.0,NaN,NaN
3,2019GQ0000369,4140.0,21,1,2,1,14.0,21,1400,6.0,0.0,1.0,NaN
4,2019GQ0000377,NaN,70,2,1,1,12.0,21,1704,6.0,0.0,NaN,NaN


## Handling Nulls

### Observations
The dataset includes information on individuals in Kentucky from the American Community Survey (ACS), with a focus on employment and education. Initial exploration shows that several employment-related columns (`OCCP`, `COW`, `WKHP`, `ESR`, `WAGP`) contain a high proportion of null values. 

### Thoughts
Nulls in employment columns likely indicate individuals who are unemployed, not in the labor force, or chose not to respond. These individuals may represent our **target population** — those who would most benefit from additional education or job training.

### Null Handling Plan
1. **Retain rows with employment-related nulls** instead of dropping them.
2. **Map nulls to meaningful labels** where appropriate:  

| Column              | Meaning                          | Null Handling Strategy           |
|---------------------|-----------------------------------|----------------------------------|
| `occupation`         | Job code                   | Null → `"Not employed"`          |
| `class_of_worker`    | Type of employer           | Null → `"Not employed"`          |
| `hours_per_week`     | Hours usually worked per week     | Null → `0` (assume not working)  |
| `wage_income`        | Total wage or salary income       | Null → `0.0` (assume no income)  |
| `education_level`    | Educational attainment            | Null → `"Unknown"`               |
| `employment_status`  | Employment status recode    | Null → `"Unknown"`               |

In [5]:
df['occupation'] = df['occupation'].fillna("Not employed")
df['class_of_worker'] = df['class_of_worker'].fillna("Not employed")
df['hours_per_week'] = df['hours_per_week'].fillna(0)
df['wage_income'] = df['wage_income'].fillna(0.0)
df['education_level'] = df['education_level'].fillna("Unknown")
df['employment_status'] = df['employment_status'].fillna("Unknown")

df.isnull().sum()

serial_number        0
occupation           0
age                  0
sex                  0
race                 0
hispanic_origin      0
education_level      0
state_fips           0
puma_area            0
employment_status    0
wage_income          0
class_of_worker      0
hours_per_week       0
dtype: int64

3. **Create a new column** (`needs_support`) to flag individuals who:
   - Have an occupation == "Not employed"  
   OR  
   - Have an employment_status code equating to unemployed, not in labor force, or unknown  
   OR  
   - Have an education_level below high school grad  

This will create a Boolean column where:  
   - 'True' means the person likely needs educational/employment support
   - 'False' means the person is likely already employed/educated

In [6]:
df['needs_support'] = (
    (df['occupation'] == 'Not employed') |
    (df['employment_status'].isin([4, 5, 6, 'Unknown'])) |
    (df['education_level'].apply(lambda x: isinstance(x, (int, float)) and x <= 15)) |
    (df['education_level'] == 'Unknown')
)

df['needs_support'].value_counts()

needs_support
True     130991
False     93229
Name: count, dtype: int64

## Mapping Coded Values
Using ACS Data Dictionary and [PUMA Crosswalk](https://www.census.gov/geographies/reference-maps/2010/geo/2010-pumas/kentucky.html)  

**Note:** PUMA (Public Use Microdata Area) codes each represent a region of Kentucky with at least 100,000 people.

This section converts coded values into descriptive labels.

In [7]:
# SEX
sex_map = {
    1: "Male",
    2: "Female"
}
df['sex'] = df['sex'].map(sex_map)

# RACE
race_map = {
    1: "White",
    2: "Black or African American",
    3: "American Indian or Alaska Native",
    4: "Alaska Native",
    5: "AIAN Tribes Specified / Other Race",
    6: "Asian",
    7: "Native Hawaiian and Other Pacific Islander",
    8: "Some Other Race",
    9: "Two or More Races"
}
df['race'] = df['race'].map(race_map)

# HISPANIC ORIGIN
hisp_map = {
    1: "Not Hispanic or Latino",
    2: "Mexican",
    3: "Puerto Rican",
    4: "Cuban",
    5: "Dominican Republic",
    6: "Central American",
    7: "South American",
    8: "Other Hispanic/Latino"
}
df['hispanic_origin'] = df['hispanic_origin'].map(hisp_map)

# EDUCATION LEVEL
education_map = {
    1: "No schooling completed",
    2: "Nursery school",
    3: "Kindergarten",
    4: "Grade 1",
    5: "Grade 2",
    6: "Grade 3",
    7: "Grade 4",
    8: "Grade 5",
    9: "Grade 6",
    10: "Grade 7",
    11: "Grade 8",
    12: "Grade 9",
    13: "Grade 10",
    14: "Grade 11",
    15: "12th grade – no diploma",
    16: "High school graduate or GED",
    17: "Some college (<1 year)",
    18: "Some college (1+ years, no degree)",
    19: "Associate's degree",
    20: "Bachelor's degree",
    21: "Master's degree",
    22: "Professional degree",
    23: "Doctorate"
}
df['education_level'] = df['education_level'].map(education_map).fillna(df['education_level'])

# STATE
df['state_fips'] = df['state_fips'].map({21: "Kentucky"})

# EMPLOYMENT STATUS
employment_map = {
    1: "Civilian, at work",
    2: "Civilian, job not at work",
    3: "Armed forces",
    4: "Unemployed",
    5: "Not in labor force",
    6: "Under 16 years old"
}
df['employment_status'] = df['employment_status'].map(employment_map).fillna(df['employment_status'])

# CLASS OF WORKER
cow_map = {
    1: "Private for-profit",
    2: "Private not-for-profit",
    3: "Local government",
    4: "State government",
    5: "Federal government",
    6: "Self-employed (not incorporated)",
    7: "Self-employed (incorporated)",
    8: "Unpaid family worker",
    9: "Not employed"
}

df['class_of_worker'] = df['class_of_worker'].map(cow_map) #Handles what was previously mapped as "Not employed" while using cow_map
df['class_of_worker'] = df['class_of_worker'].map(cow_map).fillna("Not employed") #Changes nulls back to "Not employed" after using cow_map

# EDUCATION LEVEL
edu_map = {
    "No schooling completed": "No schooling completed",
    "Nursery school": "Less than high school diploma",
    "Kindergarten": "Less than high school diploma",
    "Grade 1": "Less than high school diploma",
    "Grade 2": "Less than high school diploma",
    "Grade 3": "Less than high school diploma",
    "Grade 4": "Less than high school diploma",
    "Grade 5": "Less than high school diploma",
    "Grade 6": "Less than high school diploma",
    "Grade 7": "Less than high school diploma",
    "Grade 8": "Less than high school diploma",
    "Grade 9": "Less than high school diploma",
    "Grade 10": "Less than high school diploma",
    "Grade 11": "Less than high school diploma",
    "12th grade – no diploma": "Less than high school diploma",
    "High school graduate or GED": "High school graduate",
    "Some college (<1 year)": "Some college",
    "Some college (1+ years, no degree)": "Some college",
    "Associate's degree": "Associate's degree",
    "Bachelor's degree": "Bachelor's degree",
    "Master's degree": "Master's degree",
    "Professional degree": "Professional degree",
    "Doctorate": "Doctorate degree",
    "Unknown": "Unknown",
    "24.0": "Unknown"
}

df['education_level'] = df['education_level'].map(edu_map)

df.head()

,serial_number,occupation,age,sex,race,hispanic_origin,education_level,state_fips,puma_area,employment_status,wage_income,class_of_worker,hours_per_week,needs_support
0,2019GQ0000143,8740.0,22,Male,Black or African American,Not Hispanic or Latino,Some college,Kentucky,1903,Under 16 years old,1500.0,Not employed,40.0,True
1,2019GQ0000172,Not employed,19,Male,White,Not Hispanic or Latino,High school graduate,Kentucky,1703,Under 16 years old,0.0,Not employed,0.0,True
2,2019GQ0000249,Not employed,75,Male,White,Not Hispanic or Latino,High school graduate,Kentucky,100,Under 16 years old,0.0,Not employed,0.0,True
3,2019GQ0000369,4140.0,21,Male,Black or African American,Not Hispanic or Latino,Less than high school diploma,Kentucky,1400,Under 16 years old,0.0,Not employed,0.0,True
4,2019GQ0000377,Not employed,70,Female,White,Not Hispanic or Latino,Less than high school diploma,Kentucky,1704,Under 16 years old,0.0,Not employed,0.0,True


## Mapping Occupation Codes
Using Census Occupation Code List Crosswalk  

This will ensure occupations listed match the codes for occupations listed in my other data source.

In [8]:
# Loading crosswalk
crosswalk_df = pd.read_excel("../docs/job_code_crosswalk.xlsx", sheet_name="NEM SOC ACS crosswalk", header=4)

# Cleaning ACS Code: remove "Not employed", convert to numeric, fill to 4-digit strings
crosswalk_df["ACS Code"] = crosswalk_df["ACS Code"].replace("Not employed", pd.NA)
crosswalk_df["ACS Code"] = pd.to_numeric(crosswalk_df["ACS Code"], errors="coerce")
crosswalk_df = crosswalk_df.dropna(subset=["ACS Code"])
crosswalk_df["ACS Code"] = crosswalk_df["ACS Code"].astype(int).astype(str).str.zfill(4)

# Creating a map
acs_to_soc_map = dict(zip(crosswalk_df["ACS Code"], crosswalk_df["Matrix Occupation Code"]))

# Cleaning occupation column
df["occupation"] = df["occupation"].replace("Not employed", pd.NA)
df["occupation"] = pd.to_numeric(df["occupation"], errors="coerce")
df["occupation"] = df["occupation"].dropna().astype(int).astype(str).str.zfill(4)

# Mapping ACS → SOC
df["soc_code"] = df["occupation"].map(acs_to_soc_map)

# Removing hyphen in code (to match other dataset for later merging)
df["soc_code"] = df["soc_code"].str.replace("-", "", regex=False)
df["soc_code"] = df["soc_code"].fillna("Not employed")

# Filling unmapped values
df["soc_code"] = df["soc_code"].fillna("Not employed")

#Checking results
df["soc_code"].value_counts().head(10)

soc_code
Not employed    98945
533033           3105
412012           3105
291141           2974
537062           2957
119199           2722
252023           2542
412031           2462
411011           2261
434051           2226
Name: count, dtype: int64

# Converting to SQLite Database Table

In [9]:
# Define the path for the database and create it if it doesn't exist
db_path = "../data/cleaned_data.sqlite"
os.makedirs(os.path.dirname(db_path), exist_ok=True)

# Connect to the SQLite database
conn = sqlite3.connect(db_path)

# Export the DataFrame to a table named 'acsd_data'
df.to_sql("acsd_data", conn, if_exists="replace", index=False)

# Close the connection
conn.close()

print(f"ACS data successfully written to {db_path} in table 'acsd_data'.")

ACS data successfully written to ../data/cleaned_data.sqlite in table 'acsd_data'.
